#Variational Autoencoders (VAEs) — The "Imagination" Model

Up until now (Day 77-78), your Autoencoders were "Copy Machines." They could only reconstruct images they had already seen. Today, we build a VAE, which is a Generative Model. Instead of mapping an image to a fixed point, it maps it to a probability distribution (Mean and Variance).

This allows us to "sample" from that distribution to create brand-new digits that have never existed in the MNIST dataset.

#The "Reparameterization Trick"

The magic of a VAE is the Sampling Layer. We don't just output a number; we output a range.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. The Sampling Layer (The "Secret Sauce")
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# 2. The Encoder
latent_dim = 2 # 2D space so we can plot the "Map of Digits"
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# 3. The Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation="relu")(latent_inputs)
x = layers.Dense(784, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28))(x)
decoder = models.Model(latent_inputs, decoder_outputs, name="decoder")

# 4. The VAE Model (Connecting them)
# For VAEs, we usually define a custom train_step for the KL Divergence loss, 
# but for Day 79, we focus on the architecture.
print("Day 79: VAE Architecture with 2D Latent Space initialized.")


Day 79: VAE Architecture with 2D Latent Space initialized.


#Why 2D Latent Space?
By squeezing all digits into just two numbers ($x, y$), you can actually plot them. You’ll see that all the "1s" cluster in one corner, "7s" in another, and the space in between represents "blended" digits. By picking a random $(x, y)$ coordinate, the decoder will "imagine" a digit for you!